# EVALUATE ALL TRAINED MODELS
## Evaluate every checkpoint on train/valid/test splits and print a full comparison table.

**Chạy cell này sau khi checkpoint đã được sync về local (GCS → `/content/checkpoints/phase1/`)**

In [ ]:
# ============================================================
# EVALUATE ALL PHASE 1 CHECKPOINTS
# ============================================================

import sys, json, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, '/content/BCDA')

from training.config_phase1 import Phase1Config, config as default_config
from training.dataset_mosei import create_dataloaders
from training.evaluator import compute_metrics
from training.evaluator_emotion import compute_emotion_metrics

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

# ============================================================
# 1. Define model factory
# ============================================================
def build_model(cfg: Phase1Config):
    if cfg.model_type == 'early_fusion':
        from training.models.early_fusion import EarlyFusionLSTMRegressor
        return EarlyFusionLSTMRegressor(cfg.model)
    elif cfg.model_type == 'improved_lstm':
        from training.models.improved_lstm import ImprovedLSTMRegressor
        return ImprovedLSTMRegressor(cfg.model)
    elif cfg.model_type == 'mult':
        from training.models.mult import MulTRegressor
        return MulTRegressor(cfg.mult_model)
    raise ValueError(f'Unsupported: {cfg.model_type}')


# ============================================================
# 2. Evaluation function
# ============================================================
def evaluate_checkpoint(ckpt_path: str, task_type: str, model_type: str,
                          pkl_path: str, label: str = None) -> dict:
    """Load a checkpoint and evaluate on all splits."""
    label = label or ckpt_path
    print(f"\n{'='*60}")
    print(f"Evaluating: {label}")
    print(f"  Task: {task_type}  |  Model: {model_type}")
    print('='*60)

    # Build config
    cfg = default_config
    cfg.apply_profile('colab')
    cfg.training.task_type = task_type
    cfg.model_type = model_type
    cfg.training.batch_size = 32
    cfg.training.num_workers = 2

    if model_type == 'mult':
        cfg.mult_model.output_dim = 6 if task_type == 'emotion' else 1
        cfg.mult_model.stochastic_depth_survival = cfg.training.stochastic_depth_survival

    cfg.setup()

    # Load checkpoint
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f"  Epoch: {ckpt.get('epoch', 'N/A')}  |  "
          f"Best metric: {ckpt.get('best_metric', 'N/A')}")

    # Build model
    model = build_model(cfg)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()

    # Load data
    dataloaders = create_dataloaders(cfg, pkl_path=pkl_path)

    results = {}
    for split, loader in dataloaders.items():
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                text = batch['text'].to(device)
                audio = batch['audio'].to(device)
                vision = batch['vision'].to(device)
                labels = batch['label'].to(device)
                al = batch.get('audio_len')
                vl = batch.get('vision_len')
                preds = model(text=text, audio=audio, vision=vision,
                              audio_lengths=al.to(device) if al is not None else None,
                              vision_lengths=vl.to(device) if vl is not None else None)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_labels)

        if task_type == 'emotion':
            metrics = compute_emotion_metrics(y_true, y_pred)
        else:
            metrics = compute_metrics(y_true, y_pred)

        results[split] = metrics

        # Print summary
        if task_type == 'emotion':
            print(f"\n  [{split.upper()}] "
                  f"Mean F1={metrics.get('mean_f1', 0):.4f}  "
                  f"Mean Acc={metrics.get('mean_acc', 0):.4f}  "
                  f"Mean MAE={metrics.get('mean_mae', 0):.4f}")
        else:
            print(f"\n  [{split.upper()}] "
                  f"MAE={metrics.get('mae', 0):.4f}  "
                  f"Corr={metrics.get('corr', 0):.4f}  "
                  f"Acc2={metrics.get('acc2', 0):.4f}")

    return results

# ============================================================
# 3. Run evaluations
# ============================================================

CKPT_DIR = '/content/checkpoints/phase1'
Pkl_aligned = '/content/data/MSA-Dataset/aligned_50.pkl'
Pkl_unaligned = '/content/data/MSA-Dataset/unaligned_50.pkl'

# All checkpoints with their metadata
evaluations = []

# --- Sentiment models ---
sentiment_models = [
    {
        'ckpt': f'{CKPT_DIR}/best_model_mult.pt',
        'label': 'MulT (aligned)',
        'task': 'sentiment',
        'model': 'mult',
        'pkl': Pkl_aligned,
    },
    {
        'ckpt': f'{CKPT_DIR}/best_model_mult_unaligned.pt',
        'label': 'MulT (unaligned)',
        'task': 'sentiment',
        'model': 'mult',
        'pkl': Pkl_unaligned,
    },
    {
        'ckpt': f'{CKPT_DIR}/best_model_improved_lstm.pt',
        'label': 'Improved LSTM',
        'task': 'sentiment',
        'model': 'improved_lstm',
        'pkl': Pkl_aligned,
    },
]

# --- Emotion models ---
emotion_models = [
    {
        'ckpt': f'{CKPT_DIR}/best_model_mult_emotion.pt',
        'label': 'MulT Emotion (BCE)',
        'task': 'emotion',
        'model': 'mult',
        'pkl': Pkl_aligned,
    },
    # NOTE: best_model_mult_emotion_p1_focal.pt DIVERGED — skipped
]

for m in sentiment_models + emotion_models:
    try:
        results = evaluate_checkpoint(
            ckpt_path=m['ckpt'],
            task_type=m['task'],
            model_type=m['model'],
            pkl_path=m['pkl'],
            label=m['label'],
        )
        evaluations.append({'label': m['label'], 'task': m['task'], 'results': results})
    except FileNotFoundError:
        print(f'\nSKIPPED (not found): {m["ckpt"]}')
    except Exception as e:
        print(f'\nERROR on {m["label"]}: {e}')


# ============================================================
# 4. Comparison table
# ============================================================
print('\n' + '='*80)
print('FULL COMPARISON TABLE')
print('='*80)

# --- Sentiment table ---
sent_rows = []
for ev in evaluations:
    if ev['task'] != 'sentiment':
        continue
    label = ev['label']
    for split in ['valid', 'test']:
        r = ev['results'].get(split, {})
        sent_rows.append({
            'Model': label,
            'Split': split,
            'MAE': r.get('mae', 0),
            'Corr': r.get('corr', 0),
            'Acc2': r.get('acc2', 0),
            'Acc5': r.get('acc5', 0),
            'Acc7': r.get('acc7', 0),
            'F1': r.get('f1', 0),
        })

sent_df = pd.DataFrame(sent_rows)
if len(sent_df):
    print('\n[SENTIMENT — Regression]')
    print(sent_df.to_string(index=False, float_format='%.4f'))

# --- Emotion table ---
emo_rows = []
for ev in evaluations:
    if ev['task'] != 'emotion':
        continue
    label = ev['label']
    for split in ['valid', 'test']:
        r = ev['results'].get(split, {})
        emo_rows.append({
            'Model': label,
            'Split': split,
            'Mean_F1': r.get('mean_f1', 0),
            'Mean_Acc': r.get('mean_acc', 0),
            'Mean_MAE': r.get('mean_mae', 0),
            'Happy_F1': r.get('happy_f1', 0),
            'Sad_F1': r.get('sad_f1', 0),
            'Angry_F1': r.get('angry_f1', 0),
            'Disgust_F1': r.get('disgust_f1', 0),
            'Surprise_F1': r.get('surprise_f1', 0),
            'Fear_F1': r.get('fear_f1', 0),
        })

emo_df = pd.DataFrame(emo_rows)
if len(emo_df):
    print('\n[EMOTION — Multi-label Classification]')
    print(emo_df.to_string(index=False, float_format='%.4f'))


# ============================================================
# 5. Save results
# ============================================================
output = {
    'sentiment': sent_df.to_dict('records') if len(sent_df) else [],
    'emotion': emo_df.to_dict('records') if len(emo_df) else [],
}

out_path = '/content/drive/MyDrive/BCDA/outputs/phase1/all_evaluations.json'
import os
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nResults saved to: {out_path}')

print('\n' + '='*80)
print('DONE')
print('='*80)